# Comprehensive Distributed Training Guide

This notebook provides a comprehensive guide to distributed training using LLaMA-Factory, covering:

1. **DeepSpeed Training**: ZeRO stages and memory optimization
2. **FSDP Training**: Fully Sharded Data Parallel
3. **Ray Training**: Distributed computing framework
4. **Multi-Node Setup**: Scaling across multiple machines
5. **Performance Optimization**: Communication and memory efficiency
6. **Monitoring and Debugging**: Distributed training analysis

## Table of Contents

- [Setup and Installation](#setup-and-installation)
- [DeepSpeed Training](#deepspeed-training)
- [FSDP Training](#fsdp-training)
- [Ray Training](#ray-training)
- [Multi-Node Setup](#multi-node-setup)
- [Performance Optimization](#performance-optimization)
- [Monitoring and Debugging](#monitoring-and-debugging)
- [Best Practices](#best-practices)


## Setup and Installation

First, let's install the required dependencies for distributed training.


In [ ]:
# Install distributed training dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers[torch] datasets accelerate deepspeed
%pip install ray[tune] ray[train] ray[serve]
%pip install torch-fidelity  # For distributed training monitoring

# Import required libraries
import torch
import torch.distributed as dist
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import deepspeed
import accelerate
import ray
from ray import train, tune
import json
import os
import yaml
from typing import Dict, Any
import psutil
import GPUtil

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Check distributed setup
try:
    dist.init_process_group(backend='nccl')
    print(f"Process group initialized: {dist.get_rank()}/{dist.get_world_size()}")
    dist.destroy_process_group()
except:
    print("Single GPU/CPU setup detected")


## DeepSpeed Training

### ZeRO Stages Overview

DeepSpeed ZeRO (Zero Redundancy Optimizer) provides different stages for memory optimization:

- **ZeRO-0**: Standard data parallelism
- **ZeRO-1**: Optimizer state partitioning
- **ZeRO-2**: Optimizer + gradient partitioning
- **ZeRO-3**: Optimizer + gradient + parameter partitioning


In [ ]:
# DeepSpeed Configuration Files

# ZeRO-3 Configuration for maximum memory efficiency
ds_config_z3 = {
    "fp16": {
        "enabled": "auto",
        "auto_cast": False,
        "loss_scale": 0,
        "initial_scale_power": 16,
        "loss_scale_window": 1000,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "bf16": {
        "enabled": "auto"
    },
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "offload_param": {
            "device": "cpu",
            "pin_memory": True
        },
        "overlap_comm": True,
        "contiguous_gradients": True,
        "sub_group_size": 1000000000000,
        "reduce_bucket_size": "auto",
        "stage3_prefetch_bucket_size": "auto",
        "stage3_param_persistence_threshold": "auto",
        "stage3_max_live_parameters": 1000000000000,
        "stage3_max_reuse_distance": 1000000000000,
        "stage3_gather_16bit_weights_on_model_save": False
    },
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "steps_per_print": 10,
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "wall_clock_breakdown": False
}

# ZeRO-2 Configuration for balanced memory and speed
ds_config_z2 = {
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 200000000,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 200000000,
        "contiguous_gradients": True
    }
}

# Save configurations
with open('ds_config_z3.json', 'w') as f:
    json.dump(ds_config_z3, f, indent=2)

with open('ds_config_z2.json', 'w') as f:
    json.dump(ds_config_z2, f, indent=2)
